In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_json_path = 'rca100/cases/t089/task.json'
with open(task_json_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert is titled "inventory错误次数突增" (inventory error count surge). It occurred between 2026-03-27 18:08:44 and 2026-03-27 18:12:01 (UTC+8). The entity that triggered the alert is an APM operation named "inventory::/api/v1/inventory/{productId}".

The original code execution output of IPython Kernel is also provided below for reference:

('inventory错误次数突增', {'start': '2026-03-27T18:08:44.257676+08:00', 'end': '2026-03-27T18:12:01+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = 'rca100/cases/t089/metrics.parquet'
metrics_df = pd.read_parquet(metrics_path)

# List all available KPIs (metric names) and their associated entity sets
kpi_entity_sets = metrics_df[['metric', 'entity_set']].drop_duplicates()

kpi_entity_sets

```
Out[2]:
```
The `metrics.parquet` file contains various KPIs (metric names) associated with different entity sets. Relevant KPIs for analyzing the 'inventory' service and its operations include metrics like `error_rate`, `error`, `slow`, and `workload`, which are associated with entity sets such as `apm.operation`, `apm.instance`, and `apm.service.legacy`. These KPIs are critical for understanding service-level and operation-level performance.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0           node_ready_status            k8s.node
1310   node_memory_usage_rate            k8s.node
3144    node_not_ready_status            k8s.node
5109   node_pod_running_count            k8s.node
5240     node_disk_usage_rate            k8s.node
...                       ...                 ...
73773                workload  apm.service.legacy
73779                   error  apm.service.legacy
73782                    slow  apm.service.legacy
83464              error_rate       apm.operation
83607              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Convert 'time' column to numeric for calculations
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Filter metrics for the 'inventory' service and its operations with relevant KPIs
inventory_metrics = metrics_df[
    (metrics_df['entity_name'].str.contains('inventory', case=False)) &
    (metrics_df['metric'].isin(['error_rate', 'error', 'slow', 'workload']))
]

# Calculate global P95 thresholds for each KPI
p95_thresholds = inventory_metrics.groupby('metric')['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['metric', 'p95_threshold']

# Define the time window in UTC+8
start_time = datetime.fromisoformat('2026-03-27T18:08:44.257676+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
end_time = datetime.fromisoformat('2026-03-27T18:12:01+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert time window to microseconds for filtering
start_time_us = int(start_time.timestamp() * 1_000_000)
end_time_us = int(end_time.timestamp() * 1_000_000)

# Filter data within the specified time window
filtered_inventory_metrics = inventory_metrics[
    (inventory_metrics['time'] >= start_time_us) & 
    (inventory_metrics['time'] <= end_time_us)
]

p95_thresholds, filtered_inventory_metrics

```
Out[3]:
```
The global P95 thresholds for the selected KPIs across the entire dataset are as follows:
- `error`: 4530.0
- `error_rate`: 1.0
- `slow`: 864.0
- `workload`: 6180.0

Within the specified time window (2026-03-27 18:08:44 to 2026-03-27 18:12:01 UTC+8), the filtered data shows that the `inventory` service and its operations experienced elevated `error` counts, `error_rate`, and `workload`, with some values exceeding the calculated P95 thresholds. This indicates potential performance issues or errors in the `inventory` service during the alert period.

The original code execution output of IPython Kernel is also provided below for reference:

(       metric  p95_threshold
0       error         4530.0
1  error_rate            1.0
2        slow          864.0
3    workload         6180.0,                    time domain          entity_set                         entity_id                               entity_name      metric        value                            metric_set_id service
82770  1774606154000000    apm  apm.service.legacy                                                                   inventory       error   951.000000                                     None    None
82773  1774606154000000    apm  apm.service.legacy                                                                   inventory        slow     3.000000                                     None    None
82774  1774606154000000    apm  apm.service.legacy                                                                   inventory    workload  6063.000000                                     None    None
82844  1774606184000000    apm  apm.service.legacy                                                                   inventory       error  3283.000000                                     None    None
82847  1774606184000000    apm  apm.service.legacy                                                                   inventory        slow     3.000000                                     None    None
82848  1774606184000000    apm  apm.service.legacy                                                                   inventory    workload  5830.000000                                     None    None
82918  1774606214000000    apm  apm.service.legacy                                                                   inventory       error  5282.000000                                     None    None
82921  1774606214000000    apm  apm.service.legacy                                                                   inventory        slow     3.000000                                     None    None
82922  1774606214000000    apm  apm.service.legacy                                                                   inventory    workload  5698.000000                                     None    None
82992  1774606244000000    apm  apm.service.legacy                                                                   inventory       error  5631.000000                                     None    None
82995  1774606244000000    apm  apm.service.legacy                                                                   inventory        slow     3.000000                                     None    None
82996  1774606244000000    apm  apm.service.legacy                                                                   inventory    workload  5686.000000                                     None    None
83066  1774606274000000    apm  apm.service.legacy                                                                   inventory       error  5151.000000                                     None    None
83069  1774606274000000    apm  apm.service.legacy                                                                   inventory        slow     3.000000                                     None    None
83070  1774606274000000    apm  apm.service.legacy                                                                   inventory    workload  5207.000000                                     None    None
83140  1774606304000000    apm  apm.service.legacy                                                                   inventory       error  5077.000000                                     None    None
83143  1774606304000000    apm  apm.service.legacy                                                                   inventory        slow     3.000000                                     None    None
83144  1774606304000000    apm  apm.service.legacy                                                                   inventory    workload  5134.000000                                     None    None
83570  1774606154000000    apm       apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_rate     0.158526  apm@metric_set@apm.metric.apm.operation    None
83571  1774606184000000    apm       apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_rate     0.570262  apm@metric_set@apm.metric.apm.operation    None
83572  1774606214000000    apm       apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_rate     0.937522  apm@metric_set@apm.metric.apm.operation    None
83573  1774606244000000    apm       apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_rate     1.000000  apm@metric_set@apm.metric.apm.operation    None
83574  1774606274000000    apm       apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_rate     1.000000  apm@metric_set@apm.metric.apm.operation    None
83575  1774606304000000    apm       apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_rate     1.000000  apm@metric_set@apm.metric.apm.operation    None
83707  1774606154000000    apm        apm.instance  33c16dd0c27f7e1a60ad4bc3adcb47ba                                 inventory  error_rate     0.140250   apm@metric_set@apm.metric.apm.instance    None
83708  1774606184000000    apm        apm.instance  33c16dd0c27f7e1a60ad4bc3adcb47ba                                 inventory  error_rate     0.532495   apm@metric_set@apm.metric.apm.instance    None
83709  1774606214000000    apm        apm.instance  33c16dd0c27f7e1a60ad4bc3adcb47ba                                 inventory  error_rate     0.902734   apm@metric_set@apm.metric.apm.instance    None
83710  1774606244000000    apm        apm.instance  33c16dd0c27f7e1a60ad4bc3adcb47ba                                 inventory  error_rate     0.990561   apm@metric_set@apm.metric.apm.instance    None
83711  1774606274000000    apm        apm.instance  33c16dd0c27f7e1a60ad4bc3adcb47ba                                 inventory  error_rate     0.988597   apm@metric_set@apm.metric.apm.instance    None
83712  1774606304000000    apm        apm.instance  33c16dd0c27f7e1a60ad4bc3adcb47ba                                 inventory  error_rate     0.989023   apm@metric_set@apm.metric.apm.instance    None
83717  1774606154000000    apm        apm.instance  899ac8edd338f20ba938668e82c3cf36                inventory-6d8d574cbb-kmc9p  error_rate     0.162977   apm@metric_set@apm.metric.apm.instance    None
83718  1774606184000000    apm        apm.instance  899ac8edd338f20ba938668e82c3cf36                inventory-6d8d574cbb-kmc9p  error_rate     0.551992   apm@metric_set@apm.metric.apm.instance    None
83719  1774606214000000    apm        apm.instance  899ac8edd338f20ba938668e82c3cf36                inventory-6d8d574cbb-kmc9p  error_rate     0.923438   apm@metric_set@apm.metric.apm.instance    None
83720  1774606244000000    apm        apm.instance  899ac8edd338f20ba938668e82c3cf36                inventory-6d8d574cbb-kmc9p  error_rate     0.990042   apm@metric_set@apm.metric.apm.instance    None
83721  1774606274000000    apm        apm.instance  899ac8edd338f20ba938668e82c3cf36                inventory-6d8d574cbb-kmc9p  error_rate     0.989523   apm@metric_set@apm.metric.apm.instance    None
83722  1774606304000000    apm        apm.instance  899ac8edd338f20ba938668e82c3cf36                inventory-6d8d574cbb-kmc9p  error_rate     0.988636   apm@metric_set@apm.metric.apm.instance    None
83814  1774606154000000    apm        apm.instance  07701fda8251906f1bd87b14d34fc002                inventory-556cbf4c84-xz8nh  error_rate     0.166172   apm@metric_set@apm.metric.apm.instance    None
83815  1774606184000000    apm        apm.instance  07701fda8251906f1bd87b14d34fc002                inventory-556cbf4c84-xz8nh  error_rate     0.606760   apm@metric_set@apm.metric.apm.instance    None
83816  1774606214000000    apm        apm.instance  07701fda8251906f1bd87b14d34fc002                inventory-556cbf4c84-xz8nh  error_rate     0.955224   apm@metric_set@apm.metric.apm.instance    None
83817  1774606244000000    apm        apm.instance  07701fda8251906f1bd87b14d34fc002                inventory-556cbf4c84-xz8nh  error_rate     0.990379   apm@metric_set@apm.metric.apm.instance    None
83818  1774606274000000    apm        apm.instance  07701fda8251906f1bd87b14d34fc002                inventory-556cbf4c84-xz8nh  error_rate     0.989625   apm@metric_set@apm.metric.apm.instance    None
83819  1774606304000000    apm        apm.instance  07701fda8251906f1bd87b14d34fc002                inventory-556cbf4c84-xz8nh  error_rate     0.989024   apm@metric_set@apm.metric.apm.instance    None)```
```

In [ ]:
# In[4]:

# Load the traces.parquet file
traces_path = 'rca100/cases/t089/traces.parquet'
traces_df = pd.read_parquet(traces_path)

# Convert relevant columns to numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')

# Filter traces within the specified time window
filtered_traces = traces_df[
    (traces_df['startTime'] >= start_time_us * 1000) &  # Convert microseconds to nanoseconds
    (traces_df['endTime'] <= end_time_us * 1000)
]

# Focus on traces related to the 'inventory' service and its operations
inventory_traces = filtered_traces[
    (filtered_traces['serviceName'].str.contains('inventory', case=False)) |
    (filtered_traces['spanName'].str.contains('inventory', case=False))
]

# Identify traces with errors (statusCode != 0 or statusMessage indicating an error)
error_traces = inventory_traces[
    (inventory_traces['statusCode'] != 0) |
    (inventory_traces['statusMessage'].str.contains('error', case=False, na=False))
]

error_traces

```
Out[4]:
```
Within the specified time window (2026-03-27 18:08:44 to 2026-03-27 18:12:01 UTC+8), 4606 traces related to the 'inventory' service and its operations were identified with errors. These traces include operations such as `/api/v1/inventory/{productId}` and `GetInventory`. The errors are indicated by non-zero `statusCode` or error-related `statusMessage`. This suggests that the 'inventory' service experienced significant issues during the alert period, which aligns with the elevated error metrics observed earlier.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                       spanName  ...  __topic__      __tag__:__pack_id__  events           ip links
398081  402d0fec99aa491b3eae374efa48ff5b  e5263d12b90fbed2  85c841850caff61e    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
398082  1006a9573a97dcbbe574311ce1895a86  0336e5770db71156  611e7b154e894d2b    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
398084  1905413a0ca7fd0ef655a7990f8fe623  2bb810a4815a4a12  73fa80b30ded60dc    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
398085  7d88fe6e2df598e57cc0965d8fab6246  f8498f88f7056dca  b789e128ac184ec3    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
398086  d67630bb0697de880fe1a9f97bd9f687  6484f1b45ba4a9b3  e6a2fa4c6f054ffe    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
...                                  ...               ...               ...  ...                            ...  ...        ...                      ...     ...          ...   ...
496543  0a5faab793c0f6e34a4a33d261345f5d  182a145a35792259  0f441b48ea5b711b    1                   GetInventory  ...             58050139E7D579A6-20FC43    None         None  None
497989  82262c48e5e72a6432cc9e57b7934809  4a434346c13b5e02  b545c996344a5c55    1                   GetInventory  ...             58050139E7D579A6-20FC48    None         None  None
497998  4e92788219f2688a7bdb63aba4c00558  cde499d0de9e6efb  b1af45891816f0d0    1                   GetInventory  ...             58050139E7D579A6-20FC48    None         None  None
498006  ae6951c0382adef4bea452472f193663  2486c1139220ad4d  de3ec9c626441c75    1                   GetInventory  ...             58050139E7D579A6-20FC61    None         None  None
498009  ae6951c0382adef4bea452472f193663  23a56c7ad8e9c2cd  bf323b159d33fb4b    2  /api/v1/inventory/{productId}  ...             859609BD1D0CFE0C-20BAFC    None   10.0.2.172  None

[4606 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the logs.parquet file
logs_path = 'rca100/cases/t089/logs.parquet'
logs_df = pd.read_parquet(logs_path)

# Convert '_time_' column to numeric for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs within the specified time window
filtered_logs = logs_df[
    (logs_df['_time_'] >= start_time_us) & 
    (logs_df['_time_'] <= end_time_us)
]

# Focus on logs related to the 'inventory' service and search for error-related messages
inventory_logs_with_errors = filtered_logs[
    (filtered_logs['_container_name_'].str.contains('inventory', case=False, na=False)) &
    (filtered_logs['content'].str.contains('error', case=False, na=False))
]

inventory_logs_with_errors

```
Out[5]:
```
Within the specified time window (2026-03-27 18:08:44 to 2026-03-27 18:12:01 UTC+8), no error-related log messages were found for the 'inventory' service. This indicates that while the service experienced elevated error metrics and traces, no explicit error messages were logged during this period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```